# VWAP Events 03 — VWAP fade scalp: gross edge, and is it VWAP-specific?

Protocol: `discussions/VWAP_Events/004_claude_ve03_preregistration.md` (fixed before this run).

After a cross, the first bar stretched ≥ k·σ·√15 from VWAP → fade toward VWAP (entry 1 minute later) → exit at target / VWAP touch, stop, or 60 min.
**Primary:** k = 1, exit at VWAP touch, stop 20 bp. **G1:** hour-matched real − placebo > 0. **G2:** real gross lower bound > 7 bp.

In [ ]:
import os,sys,json,time,importlib,subprocess
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from dotenv import load_dotenv
pd.set_option('display.width',250);pd.set_option('display.max_columns',40)
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'));load_dotenv(ROOT/'.env')
from data_engineering.modeling import vwap_events as ve, vwap_fade as vf
importlib.reload(vf)
SYNTHETIC=os.getenv('VE03_SYNTHETIC','0')=='1'
ASSETS=['BTC','ETH'];fp=vf.FadeParams()
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='ve03-');OUT=Path(tmp.name)/'03';fp=vf.FadeParams(n_boot=300)
else:
    OUT=ROOT/'notebooks'/'VWAP_Events'/'03'
    BIN=Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent/'Binance'
OUT.mkdir(parents=True,exist_ok=True)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
chk=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_vwap_fade.py')],cwd=str(ROOT),env=env,capture_output=True,text=True)
print(chk.stdout[-300:]);assert chk.returncode==0,'tests failed'
print(json.dumps(vf.registration(fp),indent=1,default=str))

In [ ]:
T0=time.time();parts=[]
for i,a in enumerate(ASSETS):
    df=vf.synthetic_path_klines(n_days=90,seed=3+i) if SYNTHETIC else pd.read_parquet(BIN/f'{a}USDT_um_1m.parquet',columns=['open_time','high','low','close','volume','quote_volume'])
    parts.append(vf.run_asset(df,a,fp));print(a,f'{time.time()-T0:.0f}s')
R=pd.concat(parts,ignore_index=True)
print('trades simulated:',len(R))

## Primary decision

In [ ]:
res=vf.primary(R,fp);print(json.dumps(res,indent=1,default=float))

## Full grid (reported, not gated)

In [ ]:
G=vf.grid_table(R,fp);G.to_csv(OUT/'03_grid.csv',index=False)
for k in fp.k_grid:
    for mode in ('fade','follow'):
        g=G[(G.k==k)&(G['mode']==mode)]
        print(f'k={k} {mode}: gross bp, rows=target, cols=stop (real | placebo)')
        display(g.pivot_table(index='target',columns=['kind','stop'],values='gross_bp').round(2))
print('win rate vs S/(S+T) (real, k=1, fade, numeric targets):')
w=G[(G.kind=='real')&(G.k==1)&(G['mode']=='fade')&(G.target!='vwap')].copy()
w['theory']=w['stop']/(w['stop']+w.target.astype(float));display(w[['target','stop','win_rate','theory','mean_win','mean_loss','gross_bp','n']].round(3))
print('VWAP-touch exit (real vs placebo):');display(G[G.target=='vwap'][['kind','k','stop','gross_bp','lo','hi','win_rate','mean_win','mean_loss','n']].round(2))

## Splits of the primary cell

In [ ]:
sel=(R.k==fp.k_primary)&(R['mode']=='fade')&(R.target==str(fp.primary_target))&(R.stop==fp.primary_stop)
P=R[sel].copy();P['period']=np.where(pd.to_datetime(P.day,utc=True)>=pd.Timestamp('2026-04-01',tz='UTC'),'post_2026-04','pre_2026-04')
rows=[]
for keys,g in P.groupby(['kind','asset','anchor','period'],observed=True):
    m,lo,hi,n=vf.boot(g.pnl.to_numpy(),g.day.to_numpy(),fp.n_boot,5)
    rows.append(dict(zip(['kind','asset','anchor','period'],keys),gross_bp=m,lo=lo,hi=hi,n=n,win_rate=float((g.pnl>0).mean())))
S=pd.DataFrame(rows);display(S.round(2));S.to_csv(OUT/'03_primary_splits.csv',index=False)

## Save

In [ ]:
if not SYNTHETIC:
    (OUT/'03_decision.json').write_text(json.dumps(res,indent=1,default=float))
    (OUT/'03_manifest.json').write_text(json.dumps({'registration':vf.registration(fp),'runtime_s':time.time()-T0,'trades':int(len(R))},indent=1,default=str))
print(f'total {time.time()-T0:.0f}s')